# Notebook 02b — NRAGLS Original (O(N) Gated Linear Attention)

NRAGLS thay thế Self-Attention O(N²) bằng:
1. **Gated Linear Attention (GLA)**: Q(K⊤V) thay vì softmax(QK⊤/√d)V → O(L·d²)
2. **Gating mechanism**: σ(Wx+b) lọc nhiễu click vô tình
3. **SGLU feed-forward**: Simplified Gated Linear Unit
4. **RMSNorm**: thay LayerNorm, nhẹ hơn

News Encoder giữ nguyên như NRMS để so sánh công bằng (controlled variable).

In [ ]:
import sys, time, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, '.')
sys.path.append('/kaggle/input/datasets/neitng/utils-for-dl-major-assignment')

from utils import (
    seed_everything, TRAIN_DIR, DEV_DIR, WORK_DIR, MODEL_DIR, SEED,
    load_news, load_behaviors, parse_impressions,
    build_vocab, tokenize_to_ids,
    MINDTrainDataset, collate_train,
    compute_ranking_metrics, count_parameters
)

seed_everything(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')

EMB_DIM      = 64
NEWS_DIM     = 128
N_HEADS_NEWS = 8
N_HEADS_USER = 8
MAX_TITLE    = 30
MAX_HIST     = 30
NEG_K        = 4
BATCH_SIZE   = 64
LR           = 5e-4
WEIGHT_DECAY = 1e-5
DROPOUT      = 0.1
EPOCHS       = 5
WARMUP_RATIO = 0.1

DEVICE: cuda


In [2]:
print("Loading data...")
news_train = load_news(TRAIN_DIR)
news_dev   = load_news(DEV_DIR)
news_all   = pd.concat([news_train, news_dev]).drop_duplicates('news_id').reset_index(drop=True)
beh_train  = load_behaviors(TRAIN_DIR)
beh_dev    = load_behaviors(DEV_DIR)

print(f"News: {len(news_all)}, Train: {len(beh_train)}, Dev: {len(beh_dev)}")

Loading data...
News: 65238, Train: 156965, Dev: 73152


In [3]:
vocab = build_vocab(news_all, min_freq=2, max_len=MAX_TITLE)
print(f"Vocab size: {len(vocab)}")

all_nids = news_all['news_id'].tolist()
nid2idx = {n: i+1 for i, n in enumerate(all_nids)}
NUM_NEWS = len(nid2idx)

news_token_ids = {}
for _, row in news_all.iterrows():
    nidx = nid2idx[row['news_id']]
    text = (row['title'] or '') + ' ' + (row['abstract'] or '')
    news_token_ids[nidx] = tokenize_to_ids(text, vocab, MAX_TITLE)

news_tokens_tensor = torch.zeros(NUM_NEWS + 1, MAX_TITLE, dtype=torch.long)
for nidx, tids in news_token_ids.items():
    news_tokens_tensor[nidx] = torch.tensor(tids, dtype=torch.long)

print(f"Tokenized {NUM_NEWS} news articles")

Vocab size: 40689
Tokenized 65238 news articles


In [4]:
class AdditiveAttention(nn.Module):
    def __init__(self, dim, hidden=64):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x, mask=None):
        w = self.proj(x).squeeze(-1)
        if mask is not None:
            w = w.masked_fill(mask, -1e4)
        w = torch.softmax(w, dim=-1)
        w = torch.nan_to_num(w, nan=0.0)
        return (x * w.unsqueeze(-1)).sum(dim=1)

class NewsEncoder(nn.Module):
    """Same news encoder as NRMS (controlled variable)."""
    def __init__(self, vocab_size, emb_dim, news_dim, n_heads, dropout):
        super().__init__()
        self.word_emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.proj      = nn.Linear(emb_dim, news_dim, bias=False)
        self.norm      = nn.LayerNorm(news_dim)
        self.dropout   = nn.Dropout(dropout)
        self.mha       = nn.MultiheadAttention(news_dim, n_heads,
                                               batch_first=True, dropout=dropout)
        self.attn_pool = AdditiveAttention(news_dim)

    def forward(self, token_ids):
        mask = (token_ids == 0)
        
        # Prevent NaN gradients in MHA for all-pad sequences (e.g. dummy news)
        mha_mask = mask.clone()
        all_pad = mha_mask.all(dim=1)
        mha_mask[all_pad, 0] = False
        
        x = self.norm(self.proj(self.word_emb(token_ids)))
        x = self.dropout(x)
        x, _ = self.mha(x, x, x, key_padding_mask=mha_mask)
        x = torch.nan_to_num(x, nan=0.0)
        return self.attn_pool(x, mask)

In [5]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (lighter than LayerNorm)."""
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return self.scale * x / rms


class GatedLinearAttention(nn.Module):
    """
    Gated Linear Attention with Recency Decay: O(L · d²) complexity.

    Instead of: softmax(QK⊤/√d) V  →  O(L² · d)
    Computes:   φ(Q) · (φ(K)⊤ · V)  →  O(L · d²)

    Original NRAGLS Features:
    - Content-based gating: g = σ(Wx+b) filters noisy clicks
    """
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        assert dim % n_heads == 0

        self.W_q = nn.Linear(dim, dim)
        self.W_k = nn.Linear(dim, dim)
        self.W_v = nn.Linear(dim, dim)
        self.W_o = nn.Linear(dim, dim)

        # Content-based gating (from NRAGLS paper)
        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid()
        )

        self.dropout = nn.Dropout(dropout)

    def _feature_map(self, x):
        """Kernel feature map φ(x) = elu(x) + 1 (ensures non-negativity)."""
        return F.elu(x) + 1.0

    def forward(self, x, mask=None):
        B, L, D = x.shape
        H, d = self.n_heads, self.head_dim

        # Project Q, K, V
        Q = self._feature_map(self.W_q(x)).view(B, L, H, d)
        K = self._feature_map(self.W_k(x)).view(B, L, H, d)
        V = self.W_v(x).view(B, L, H, d)

        # Content-based gate (from paper)
        g = self.gate(x).view(B, L, H, d)  # (B, L, H, d)

        K = K * g
        V = V * g

        # Mask padding positions
        if mask is not None:
            pad_mask = (~mask).float().view(B, L, 1, 1)  # 1=valid, 0=pad
            K = K * pad_mask
            V = V * pad_mask

        # Linear Attention: Q @ (K⊤ @ V) → O(L · d²)
        KV = torch.einsum('blhd,blhe->bhde', K, V)  # (B, H, d, d)
        out = torch.einsum('blhd,bhde->blhe', Q, KV)  # (B, L, H, d)

        # Normalize by sum of keys — clamp prevents NaN in backward
        K_sum = K.sum(dim=1)  # (B, H, d)
        denom = torch.einsum('blhd,bhd->blh', Q, K_sum).unsqueeze(-1)
        denom = denom.clamp(min=0.1)  # strong clamp — avoids 1/~0 gradient explosion
        out = out / denom

        out = torch.nan_to_num(out.reshape(B, L, D), nan=0.0)
        return self.dropout(self.W_o(out))


class SGLU(nn.Module):
    """Simplified Gated Linear Unit feed-forward."""
    def __init__(self, dim, expansion=2, dropout=0.1):  # expansion=2 for smaller model
        super().__init__()
        hidden = dim * expansion
        self.W1 = nn.Linear(dim, hidden)
        self.W2 = nn.Linear(dim, hidden)
        self.W_out = nn.Linear(hidden, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.W_out(F.silu(self.W1(x)) * self.W2(x)))


class GLAUserEncoderLayer(nn.Module):
    """Single Gated Linear Attention layer + SGLU + RMSNorm."""
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.gla = GatedLinearAttention(dim, n_heads, dropout)
        self.norm2 = RMSNorm(dim)
        self.ffn = SGLU(dim, expansion=4, dropout=dropout)

    def forward(self, x, mask=None):
        x = x + self.gla(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


class GLAUserEncoder(nn.Module):
    """User Encoder with Gated Linear Attention (replaces NRMS Self-Attention)."""
    def __init__(self, news_dim, n_heads, n_layers=2, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            GLAUserEncoderLayer(news_dim, n_heads, dropout)
            for _ in range(n_layers)
        ])
        self.attn_pool = AdditiveAttention(news_dim)

    def forward(self, news_vecs, mask=None):
        x = news_vecs
        for layer in self.layers:
            x = layer(x, mask)
        return self.attn_pool(x, mask)

In [6]:
class NRAGLS(nn.Module):
    def __init__(self, vocab_size, emb_dim, news_dim, n_heads_news,
                 n_heads_user, dropout, news_tokens_lut, n_user_layers=2):
        super().__init__()
        # Same news encoder as NRMS (controlled variable)
        self.news_encoder = NewsEncoder(vocab_size, emb_dim, news_dim,
                                        n_heads_news, dropout)
        # GLA-based user encoder (the key innovation)
        self.user_encoder = GLAUserEncoder(news_dim, n_heads_user,
                                           n_layers=n_user_layers,
                                           dropout=dropout)
        self.register_buffer('news_lut', news_tokens_lut)
        self.news_dim = news_dim

    def get_news_tokens(self, nid_indices):
        return self.news_lut[nid_indices]

    def encode_news(self, nid_indices):
        tokens = self.get_news_tokens(nid_indices)
        shape = tokens.shape
        if len(shape) > 2:
            flat = tokens.view(-1, shape[-1])
            vecs = self.news_encoder(flat)
            return vecs.view(*shape[:-1], -1)
        return self.news_encoder(tokens)

    def encode_user(self, hist_indices):
        mask = (hist_indices == 0)
        news_vecs = self.encode_news(hist_indices)
        return self.user_encoder(news_vecs, mask)

    def forward(self, hist, pos, neg):
        u = self.encode_user(hist)
        p_vec = self.encode_news(pos)
        n_vec = self.encode_news(neg)
        pos_score = (u * p_vec).sum(-1, keepdim=True)
        neg_score = (u.unsqueeze(1) * n_vec).sum(-1)
        return torch.cat([pos_score, neg_score], dim=1)

    def score_candidates(self, hist, cand_indices):
        u = self.encode_user(hist)
        c_vec = self.encode_news(cand_indices)
        return (u * c_vec).sum(-1)

In [7]:
model = NRAGLS(
    vocab_size=len(vocab),
    emb_dim=EMB_DIM,
    news_dim=NEWS_DIM,
    n_heads_news=N_HEADS_NEWS,
    n_heads_user=N_HEADS_USER,
    dropout=DROPOUT,
    news_tokens_lut=news_tokens_tensor,
    n_user_layers=2,
).to(DEVICE)

n_params = count_parameters(model)
print(f"NRAGLS parameters: {n_params:,} ({n_params/1e6:.2f}M)")

NRAGLS parameters: 3,256,386 (3.26M)


In [8]:
print("Building training dataset...")
train_ds = MINDTrainDataset(beh_train, nid2idx, max_hist=MAX_HIST,
                            neg_k=NEG_K, max_rows=150000)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      collate_fn=lambda b: collate_train(b, MAX_HIST),
                      num_workers=2, pin_memory=True)
print(f"Training samples: {len(train_ds)}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.CrossEntropyLoss()

# Linear warmup + cosine decay scheduler
total_steps = len(train_dl) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

epoch_times = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    t0 = time.time()
    for H, P, N in train_dl:
        H, P, N = H.to(DEVICE), P.to(DEVICE), N.to(DEVICE)
        optimizer.zero_grad()
        logits = model(H, P, N)
        target = torch.zeros(logits.size(0), dtype=torch.long, device=DEVICE)
        loss = loss_fn(logits, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

    dt = time.time() - t0
    epoch_times.append(dt)
    print(f"Epoch {epoch}/{EPOCHS} | loss={np.mean(losses):.5f} | time={dt:.1f}s")

Building training dataset...
Training samples: 150000
Epoch 1/5 | loss=1.50008 | time=116.6s
Epoch 2/5 | loss=1.39450 | time=119.7s
Epoch 3/5 | loss=1.33358 | time=119.7s
Epoch 4/5 | loss=1.26526 | time=119.9s
Epoch 5/5 | loss=1.20749 | time=119.9s


In [9]:
print("\nEvaluating on dev set...")
from utils import evaluate_model
metrics = evaluate_model(model, beh_dev, nid2idx, news_token_ids,
                         max_hist=MAX_HIST, device=DEVICE)
print("=" * 50)
print("NRAGLS Original Results:")
for k, v in metrics.items():
    print(f"  {k}: {v:.6f}")
print("=" * 50)


Evaluating on dev set...
NRAGLS Original Results:
  AUC: 0.591961
  MRR: 0.313144
  nDCG@5: 0.292872
  nDCG@10: 0.356165


In [10]:
results = {
    'model': 'NRAGLS',
    'type': 'original',
    'complexity': 'O(L * d^2)',
    'parameters': n_params,
    'metrics': metrics,
    'epoch_times_sec': epoch_times,
    'avg_epoch_time_sec': float(np.mean(epoch_times)),
    'hyperparams': {
        'emb_dim': EMB_DIM, 'news_dim': NEWS_DIM,
        'n_heads_news': N_HEADS_NEWS, 'n_heads_user': N_HEADS_USER,
        'max_title': MAX_TITLE, 'max_hist': MAX_HIST,
        'neg_k': NEG_K, 'batch_size': BATCH_SIZE,
        'lr': LR, 'epochs': EPOCHS, 'dropout': DROPOUT,
        'n_user_layers': 2,
    }
}

torch.save(model.state_dict(), MODEL_DIR / 'nragls_original.pt')
with open(WORK_DIR / 'nragls_original_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nModel saved to {MODEL_DIR / 'nragls_original.pt'}")
print(f"Results saved to {WORK_DIR / 'nragls_original_results.json'}")


Model saved to /kaggle/working/dl_results/models/nragls_original.pt
Results saved to /kaggle/working/dl_results/nragls_original_results.json
